In [86]:
import pandas as pd
import numpy as np

In [87]:
drivers = pd.read_csv("../data/raw/drivers.csv", na_values=["\\N"])
constructors = pd.read_csv("../data/raw/constructors.csv", na_values=["\\N"])
races = pd.read_csv("../data/raw/races.csv", na_values=["\\N"])
results = pd.read_csv("../data/raw/results.csv", na_values=["\\N"])
qualifying = pd.read_csv("../data/raw/qualifying.csv", na_values=["\\N"])
status = pd.read_csv("../data/raw/status.csv", na_values=["\\N"])

In [88]:
datasets = {"drivers":drivers, "constructors":constructors, "races":races, "results":results, "quali":qualifying, "status":status}
for name, df in datasets.items():
    print(name + ': ' + str(df.shape))

drivers: (861, 9)
constructors: (212, 5)
races: (1125, 18)
results: (26759, 18)
quali: (10494, 9)
status: (139, 2)


### Filtering datasets to include Formula 1 races from 2014 to 2024.
Only race results and qualifying records associated with these races are retained.

In [89]:
races = races[(races["year"] >= 2014) & (races["year"] <= 2024)]

In [90]:
results = results[results["raceId"].isin(races["raceId"])]
qualifying = qualifying[qualifying["raceId"].isin(races["raceId"])]

In [91]:
print("Races:", races.shape)
print("Results:", results.shape)
print("Qualifying:", qualifying.shape)

Races: (228, 18)
Results: (4626, 18)
Qualifying: (4610, 9)


### Unnecessary columns are removed to simplify the datasets and retain only the information relevant to driver and constructor performance analysis.
Also, columns with high percentage of missing values wont be used.

In [92]:
drivers = drivers.drop(columns=["driverRef", "number", "dob", "url"])

In [93]:
constructors = constructors.drop(columns=["constructorRef", "url"])

In [94]:
races = races.drop(columns=[ "circuitId", "time", "url", "fp1_date", "fp1_time", "fp2_date", "fp2_time", "fp3_date", "fp3_time",
    "quali_date", "quali_time", "sprint_date", "sprint_time"])

In [95]:
results = results.drop(columns=[ "number", "position", "positionText", "time", "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed"])

In [96]:
qualifying = qualifying.drop(columns=["number"])

In [97]:
for name, df in datasets.items():
    print(name, df.shape)
    print(df.columns.tolist())
    print()

drivers (861, 9)
['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob', 'nationality', 'url']

constructors (212, 5)
['constructorId', 'constructorRef', 'name', 'nationality', 'url']

races (1125, 18)
['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time']

results (26759, 18)
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']

quali (10494, 9)
['qualifyId', 'raceId', 'driverId', 'constructorId', 'number', 'position', 'q1', 'q2', 'q3']

status (139, 2)
['statusId', 'status']



In [98]:
datasets = {"drivers":drivers, "constructors":constructors, "races":races, "results":results, "quali":qualifying, "status":status}
for name, df in datasets.items():
    print("Missing values in dataset " + name.upper() + ", per column: ")
    print(df.isnull().sum())
    print()

Missing values in dataset DRIVERS, per column: 
driverId         0
code           757
forename         0
surname          0
nationality      0
dtype: int64

Missing values in dataset CONSTRUCTORS, per column: 
constructorId    0
name             0
nationality      0
dtype: int64

Missing values in dataset RACES, per column: 
raceId    0
year      0
round     0
name      0
date      0
dtype: int64

Missing values in dataset RESULTS, per column: 
resultId         0
raceId           0
driverId         0
constructorId    0
grid             0
positionOrder    0
points           0
laps             0
statusId         0
dtype: int64

Missing values in dataset QUALI, per column: 
qualifyId           0
raceId              0
driverId            0
constructorId       0
position            0
q1                 51
q2               1222
q3               2397
dtype: int64

Missing values in dataset STATUS, per column: 
statusId    0
status      0
dtype: int64



In [99]:
results["win"] = (results["positionOrder"] == 1).astype(int)

In [100]:
results["podium"] = (results["positionOrder"] <= 3).astype(int)

In [101]:
results["positions_gained"] = results["grid"] - results["positionOrder"]

In [102]:
results.head()

,resultId,raceId,driverId,constructorId,grid,positionOrder,points,laps,statusId,win,podium,positions_gained
22127,22130,900,3,131,3,1,25.0,57,1,1,1,2
22128,22131,900,825,1,4,2,18.0,57,1,0,1,2
22129,22132,900,18,1,10,3,15.0,57,1,0,1,7
22130,22133,900,4,6,5,4,12.0,57,1,0,0,1
22131,22134,900,822,3,15,5,10.0,57,1,0,0,10


In [103]:
races['name'].unique()

<StringArray>
[      'Australian Grand Prix',        'Malaysian Grand Prix',
          'Bahrain Grand Prix',          'Chinese Grand Prix',
          'Spanish Grand Prix',           'Monaco Grand Prix',
         'Canadian Grand Prix',         'Austrian Grand Prix',
          'British Grand Prix',           'German Grand Prix',
        'Hungarian Grand Prix',          'Belgian Grand Prix',
          'Italian Grand Prix',        'Singapore Grand Prix',
         'Japanese Grand Prix',          'Russian Grand Prix',
    'United States Grand Prix',        'Brazilian Grand Prix',
        'Abu Dhabi Grand Prix',          'Mexican Grand Prix',
         'European Grand Prix',       'Azerbaijan Grand Prix',
           'French Grand Prix',          'Styrian Grand Prix',
 '70th Anniversary Grand Prix',           'Tuscan Grand Prix',
            'Eifel Grand Prix',       'Portuguese Grand Prix',
   'Emilia Romagna Grand Prix',          'Turkish Grand Prix',
           'Sakhir Grand Prix',          

In [104]:
countries = {
    "Australian Grand Prix": "Australia",       "Malaysian Grand Prix": "Malaysia",
    "Bahrain Grand Prix": "Bahrain",        "Chinese Grand Prix": "China",
    "Spanish Grand Prix": "Spain",      "Monaco Grand Prix": "Monaco",
    "Canadian Grand Prix": "Canada",        "Austrian Grand Prix": "Austria",
    "British Grand Prix": "United Kingdom",     "German Grand Prix": "Germany",
    "Hungarian Grand Prix": "Hungary",      "Belgian Grand Prix": "Belgium",
    "Italian Grand Prix": "Italy",      "Singapore Grand Prix": "Singapore",
    "Japanese Grand Prix": "Japan",     "Russian Grand Prix": "Russia",
    "United States Grand Prix": "United States",        "Brazilian Grand Prix": "Brazil",
    "Abu Dhabi Grand Prix": "United Arab Emirates",     "Mexican Grand Prix": "Mexico",
    "European Grand Prix": "Azerbaijan",        "Azerbaijan Grand Prix": "Azerbaijan",
    "French Grand Prix": "France",      "Styrian Grand Prix": "Austria",
    "70th Anniversary Grand Prix": "United Kingdom",        "Tuscan Grand Prix": "Italy",
    "Eifel Grand Prix": "Germany",      "Portuguese Grand Prix": "Portugal",
    "Emilia Romagna Grand Prix": "Italy",       "Turkish Grand Prix": "Turkey",
    "Sakhir Grand Prix": "Bahrain",     "Qatar Grand Prix": "Qatar",
    "Dutch Grand Prix": "Netherlands",      "Mexico City Grand Prix": "Mexico",
    "São Paulo Grand Prix": "Brazil",       "Saudi Arabian Grand Prix": "Saudi Arabia",
    "Miami Grand Prix": "United States",        "Las Vegas Grand Prix": "United States"
}

In [105]:
races["race country"] = races["name"].map(countries)

In [106]:
print(races["race country"].isnull().sum())

0


In [107]:
print(races[["name", "race country"]].head(10))

                      name    race country
897  Australian Grand Prix       Australia
898   Malaysian Grand Prix        Malaysia
899     Bahrain Grand Prix         Bahrain
900     Chinese Grand Prix           China
901     Spanish Grand Prix           Spain
902      Monaco Grand Prix          Monaco
903    Canadian Grand Prix          Canada
904    Austrian Grand Prix         Austria
905     British Grand Prix  United Kingdom
906      German Grand Prix         Germany


In [108]:
drivers["driver name"] = drivers["forename"] + " " + drivers["surname"]

In [109]:
drivers.head()

,driverId,code,forename,surname,nationality,driver name
0,1,HAM,Lewis,Hamilton,British,Lewis Hamilton
1,2,HEI,Nick,Heidfeld,German,Nick Heidfeld
2,3,ROS,Nico,Rosberg,German,Nico Rosberg
3,4,ALO,Fernando,Alonso,Spanish,Fernando Alonso
4,5,KOV,Heikki,Kovalainen,Finnish,Heikki Kovalainen


In [110]:
drivers.to_csv("../data/processed/drivers.csv", index=False)
constructors.to_csv("../data/processed/constructors.csv", index=False)
races.to_csv("../data/processed/races.csv", index=False)
results.to_csv("../data/processed/results.csv", index=False)
qualifying.to_csv("../data/processed/qualifying.csv", index=False)
status.to_csv("../data/processed/status.csv", index=False)